# Multi-Image Inference with LLaVA 1.5 using Hugging Face Transformers
This notebook demonstrates how to pass and analyze multiple images in a single prompt using the LLaVA 1.5 model.

### Cell 1: Install Dependencies

In [1]:
# Run this cell to install required libraries if you haven't already
!pip install -q transformers accelerate bitsandbytes pillow requests

### Cell 2: Import Libraries & Initialize Model

In [2]:
import requests
from PIL import Image
import torch
from transformers import LlavaProcessor, LlavaForConditionalGeneration

# Initialize model with FP16 to fit on standard GPU runtimes (like Google Colab)
model_id = "llava-hf/llava-1.5-7b-hf"
processor = LlavaProcessor.from_pretrained(model_id)

model = LlavaForConditionalGeneration.from_pretrained(
    model_id, 
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="auto"
)

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.0
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


ImportError: 
CLIPImageProcessorFast requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.


### Cell 3: Load and Display Images

In [ ]:
# Download sample images
url1 = "https://images.unsplash.com/photo-1543466835-00a7907e9de1"  # Dog
url2 = "https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba"  # Cat

image1 = Image.open(requests.get(url1, stream=True).raw).convert("RGB")
image2 = Image.open(requests.get(url2, stream=True).raw).convert("RGB")

# Display images inline side-by-side in your notebook
display(image1.resize((300, 300)))
display(image2.resize((300, 300)))

### Cell 4: Run Inference and Print Result

In [ ]:
# Construct prompt matching the order of images
prompt = "USER: <image>\n<image>\nCompare these two animals. What are they?\nASSISTANT:"

# Process and run inference
inputs = processor(text=prompt, images=[image1, image2], return_tensors="pt").to("cuda")
output = model.generate(**inputs, max_new_tokens=150)

# Decode output
response = processor.decode(output, skip_special_tokens=True)

# Print clean output without rewriting the prompt
print(response.split("ASSISTANT:")[-1].strip())